In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
import time

# Load frozen embedding table
embedding_table = np.load("tokenizer_v3/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)

VOCAB_SIZE    = embedding_tensor.shape[0]   # 4096
MINILM_DIM    = embedding_tensor.shape[1]   # 384
print(f"Vocab size : {VOCAB_SIZE}")
print(f"MiniLM dim : {MINILM_DIM}")


Vocab size : 16384
MiniLM dim : 384


In [2]:
import torch.nn.functional as F 

DIM      = 384
EXPANDED_DIM = 256
N_HEADS  = 8
N_LAYERS = 24
FFN_DIM  = 512
HEAD_DIM = DIM // N_HEADS 

def sinusoidal_encoding(seq_len, dim, device):
    pe       = torch.zeros(seq_len, dim, device=device)
    position = torch.arange(seq_len, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # (seq_len, dim)


class MicroLM(nn.Module):
    """
    MicroLM using PyTorch built-in TransformerEncoderLayer.

    Flow:
      token_ids
        -> frozen embedding_table lookup
        -> expander: 384 -> 512
        -> sinusoidal positional encoding
        -> N transformer encoder layers with causal mask
        -> final LayerNorm
        -> output_head
    """

    def __init__(self):
        super().__init__()

        # Frozen MiniLM-style embedding table
        self.register_buffer("embedding_table", embedding_tensor)

        # Expander 384 -> 512
        self.expander = nn.Sequential(
            nn.Linear(MINILM_DIM, EXPANDED_DIM),
            nn.LayerNorm(EXPANDED_DIM),
        )

        # Built-in transformer block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EXPANDED_DIM,
            nhead=N_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=0.1,
            activation="gelu",
            batch_first=True,     # input/output shape: (B, T, D)
            norm_first=True       # pre-norm, same style as your original block
            
        )

        self.blocks = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=N_LAYERS
        )

        self.norm = nn.LayerNorm(EXPANDED_DIM)
        self.output_head = nn.Linear(EXPANDED_DIM, VOCAB_SIZE, bias=False)

    def make_causal_mask(self, T, device):
        """
        PyTorch Transformer expects mask shape: (T, T)

        True means blocked when using bool mask.
        So upper triangle = True.
        """
        return torch.triu(
            torch.ones(T, T, device=device, dtype=torch.bool),
            diagonal=1
        )

    def forward(self, token_ids):
        B, T = token_ids.shape

        # (B, T, 384)
        x = self.embedding_table[token_ids]

        # (B, T, 512)
        x = self.expander(x)

        # Add positional encoding
        x = x + sinusoidal_encoding(T, EXPANDED_DIM, token_ids.device)

        # Causal mask for GPT-style left-to-right generation
        causal_mask = self.make_causal_mask(T, token_ids.device)

        # (B, T, 512)
        x = self.blocks(x, mask=causal_mask)

        x = self.norm(x)

        # (B, T, VOCAB_SIZE)
        logits = self.output_head(x)

        return logits

In [3]:
model     = MicroLM()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")

# print(f"\nParam breakdown:")
# for name, p in model.named_parameters():
#     if p.requires_grad:
#         print(f"  {name:55s} {p.numel():>10,}")

# Forward pass
x      = torch.randint(0, VOCAB_SIZE, (2, 32))
logits = model(x)
print(f"\nInput  : {x.shape}")
print(f"Output : {logits.shape}")

/tmp/ipykernel_10196/1262110867.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(


Trainable params : 16,944,384
Frozen params    : 0  (embedding table)
Total params     : 16,944,384

Input  : torch.Size([2, 32])
Output : torch.Size([2, 32, 16384])


In [4]:
# Freeze expander
for param in model.expander.parameters():
    param.requires_grad = False

In [5]:
total     = sum(p.numel() for p in model.expander.parameters())
trainable = sum(p.numel() for p in model.expander.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")


Trainable params : 0
Frozen params    : 99,072  (embedding table)
Total params     : 99,072


In [6]:
(2*268435456)/10603776

50.63016344366384

In [6]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
import json

# ──────────────────────────────────────────────────────────────────────────────
# tokenizer
# ──────────────────────────────────────────────────────────────────────────────
tok = Tokenizer.from_file("tokenizer_v3/tokenizer.json")

PAD_ID       = 0
UNK_ID       = 1
BOS_ID       = 2
EOS_ID       = 3
SYSTEM_ID    = 4
USER_ID      = 5
ASSISTANT_ID = 6

ROLE_TO_ID = {
    "system": SYSTEM_ID,
    "user": USER_ID,
    "assistant": ASSISTANT_ID,
}

def encode(text):
    return tok.encode(text).ids


# ──────────────────────────────────────────────────────────────────────────────
# convert messages → token ids
# ──────────────────────────────────────────────────────────────────────────────
def messages_to_tokens(messages):
    ids = [BOS_ID]

    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        ids.append(ROLE_TO_ID[role])
        ids.extend(encode(content))

    ids.append(EOS_ID)

    return ids


# ──────────────────────────────────────────────────────────────────────────────
# dataset
# ──────────────────────────────────────────────────────────────────────────────
MAX_SEQ = 512

class ConversationDataset(Dataset):
    def __init__(self, jsonl_path):
        self.samples = []

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)

                ids = messages_to_tokens(row["messages"])

                # truncate
                ids = ids[:MAX_SEQ]

                if len(ids) < 2:
                    continue

                self.samples.append(torch.tensor(ids, dtype=torch.long))

        print(f"Loaded {len(self.samples):,} conversations")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ──────────────────────────────────────────────────────────────────────────────
# collate
# full-sequence training (your current preference)
# ──────────────────────────────────────────────────────────────────────────────
def collate_fn(batch):
    max_len = max(len(x) for x in batch)

    x_batch = []
    y_batch = []

    for seq in batch:
        x = seq[:-1]
        y = seq[1:]

        pad_len = max_len - 1 - len(x)

        if pad_len > 0:
            x = torch.cat([
                x,
                torch.full((pad_len,), PAD_ID, dtype=torch.long)
            ])

            y = torch.cat([
                y,
                torch.full((pad_len,), -100, dtype=torch.long)
            ])

        x_batch.append(x)
        y_batch.append(y)

    x_batch = torch.stack(x_batch)
    y_batch = torch.stack(y_batch)

    return x_batch, y_batch


# ──────────────────────────────────────────────────────────────────────────────
# dataset + loader
# ──────────────────────────────────────────────────────────────────────────────
dataset = ConversationDataset("conversation.jsonl")


Loaded 201,526 conversations


In [7]:
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)


In [8]:
# ──────────────────────────────────────────────────────────────────────────────
# model
# your existing model class must already exist:
# class MicroLM(...)
# ──────────────────────────────────────────────────────────────────────────────
device = "cuda"

model = MicroLM().to(device)
model

/tmp/ipykernel_10196/1262110867.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(


MicroLM(
  (expander): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (blocks): TransformerEncoder(
    (layers): ModuleList(
      (0-23): 24 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (output_head): Linear(in_features=256, out_features=16384, bias=

In [9]:
# ──────────────────────────────────────────────────────────────────────────────
# load checkpoint
# ──────────────────────────────────────────────────────────────────────────────
checkpoint = torch.load(
    "checkpoints_v7/epoch_1.pt",
    map_location=device
)

model.load_state_dict(checkpoint["model"])

print("Loaded checkpoint successfully")

Loaded checkpoint successfully


In [10]:
# Freeze expander
for param in model.expander.parameters():
    param.requires_grad = False
    
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table)")
print(f"Total params     : {total:,}")

Trainable params : 16,845,312
Frozen params    : 99,072  (embedding table)
Total params     : 16,944,384


In [11]:
# ──────────────────────────────────────────────────────────────────────────────
# optimizer
# lower LR for SFT
# ──────────────────────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01
)


# optionally restore optimizer
# if "optimizer_state_dict" in checkpoint:
#     optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
#     print("Loaded optimizer state")


# ──────────────────────────────────────────────────────────────────────────────
# training loop
# ──────────────────────────────────────────────────────────────────────────────
EPOCHS = 3

model.train()

for epoch in range(EPOCHS):

    running_loss = 0.0

    for step, (x, y) in enumerate(loader):

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)

        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1),
            ignore_index=-100
        )

        optimizer.zero_grad(set_to_none=True)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        running_loss += loss.item()

        if step % 100 == 0:
            avg = running_loss / (step + 1)

            print(
                f"epoch={epoch} "
                f"step={step} "
                f"loss={avg:.4f}"
            )

    # save checkpoint
    save_path = f"checkpoints_sft_v7/epoch_{epoch}.pt"

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
    }, save_path)

    print(f"saved → {save_path}")

epoch=0 step=0 loss=7.4368
epoch=0 step=100 loss=5.1603
epoch=0 step=200 loss=4.8690
epoch=0 step=300 loss=4.7145
epoch=0 step=400 loss=4.6042
epoch=0 step=500 loss=4.5256
epoch=0 step=600 loss=4.4622
epoch=0 step=700 loss=4.4036
epoch=0 step=800 loss=4.3532
epoch=0 step=900 loss=4.3090
epoch=0 step=1000 loss=4.2695
epoch=0 step=1100 loss=4.2348
epoch=0 step=1200 loss=4.2021
epoch=0 step=1300 loss=4.1704
epoch=0 step=1400 loss=4.1411
epoch=0 step=1500 loss=4.1154
epoch=0 step=1600 loss=4.0910
epoch=0 step=1700 loss=4.0694
epoch=0 step=1800 loss=4.0469
epoch=0 step=1900 loss=4.0259
epoch=0 step=2000 loss=4.0073
epoch=0 step=2100 loss=3.9881
epoch=0 step=2200 loss=3.9700
epoch=0 step=2300 loss=3.9536
epoch=0 step=2400 loss=3.9370
epoch=0 step=2500 loss=3.9216
epoch=0 step=2600 loss=3.9070
epoch=0 step=2700 loss=3.8938
epoch=0 step=2800 loss=3.8795
epoch=0 step=2900 loss=3.8663
epoch=0 step=3000 loss=3.8535
epoch=0 step=3100 loss=3.8405
epoch=0 step=3200 loss=3.8279
epoch=0 step=3300 loss

RuntimeError: Parent directory checkpoints_sft_v7 does not exist.

In [12]:

PAD_ID       = 0
BOS_ID       = 2
EOS_ID       = 3
SYSTEM_ID    = 4
USER_ID      = 5
ASSISTANT_ID = 6

ID_TO_SPECIAL = {
    PAD_ID: "<PAD>",
    BOS_ID: "<BOS>",
    EOS_ID: "<EOS>",
    SYSTEM_ID: "<SYSTEM>",
    USER_ID: "<USER>",
    ASSISTANT_ID: "<ASSISTANT>",
}

def encode(text):
    return tok.encode(text).ids

def decode(ids):
    clean = []
    for i in ids:
        if i in ID_TO_SPECIAL:
            continue
        clean.append(i)
    return tok.decode(clean)


# ── load model ───────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

model.eval()

print("Model loaded.")


# ── build conversation prompt ────────────────────────────────────────────────
def build_prompt(messages):
    ids = [BOS_ID]

    for msg in messages:
        role = msg["role"]
        content = msg["content"]

        if role == "system":
            ids.append(SYSTEM_ID)
        elif role == "user":
            ids.append(USER_ID)
        elif role == "assistant":
            ids.append(ASSISTANT_ID)
        else:
            raise ValueError(f"Unknown role: {role}")

        ids.extend(encode(content))

    # add assistant tag so model knows it should answer now
    ids.append(ASSISTANT_ID)

    return ids


# ── generation ───────────────────────────────────────────────────────────────
@torch.no_grad()
def generate_reply(
    messages,
    max_new_tokens=120,
    temperature=0.8,
    top_k=1,
):
    ids = build_prompt(messages)

    for _ in range(max_new_tokens):
        x = torch.tensor([ids], dtype=torch.long, device=device)

        logits = model(x)
        next_logits = logits[0, -1, :]

        # prevent generating weird control tokens too early if you want
        next_logits[PAD_ID] = -float("inf")
        next_logits[BOS_ID] = -float("inf")
        next_logits[USER_ID] = -float("inf")
        next_logits[SYSTEM_ID] = -float("inf")

        next_logits = next_logits / temperature

        if top_k is not None:
            values, indices = torch.topk(next_logits, top_k)
            filtered = torch.full_like(next_logits, -float("inf"))
            filtered[indices] = values
            next_logits = filtered

        probs = torch.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()

        if next_id == EOS_ID:
            break

        # stop if model starts another role
        if next_id in [USER_ID, SYSTEM_ID, ASSISTANT_ID]:
            break

        ids.append(next_id)

    # only decode generated assistant part
    prompt_len = len(build_prompt(messages))
    generated = ids[prompt_len:]

    return decode(generated), ids


Model loaded.


In [13]:
%%time
# ── test single turn ─────────────────────────────────────────────────────────
messages = [
    {
        "role": "system",
        "content": "Your name is kunal. you are girl"
    },
    {
        "role": "user",
        "content": "hi "
    }
]

reply, token_ids = generate_reply(messages)

print("\nAssistant:")
print(reply)


Assistant:
I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot.  I am a robot. 
CPU times: user 1.38 s, sys: 488 ms, total: 1.87 s
Wall time: 1.87 s
